# Anomaly & Outlier Detection

Companion notebook for the [Anomaly Detection lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/14-anomaly-detection).

**The idea in one sentence.** Anomalies are *rare* (fraud, defects, intrusions), so the
data is extremely imbalanced — which makes **accuracy useless** (a "call everything normal"
detector scores 98%+) and forces you to score anomaly-ness and evaluate at a chosen
**alert rate** with precision/recall.

Three scoring methods, from scratch:

- **Z-score:** distance from the mean in standard deviations (assumes roughly Gaussian).
- **kNN distance:** distance to the k-th neighbour (density-based, no distribution
  assumption).
- **Isolation:** anomalies are *easier to isolate* with random splits (shorter tree depth).

We build all three, **validate that accuracy misleads while recall-at-alert-rate is
meaningful**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## 1 — Data: mostly normal, a few anomalies

A dense normal cluster plus a handful of scattered anomalies — the typical extreme imbalance.

In [ ]:
normal = rng.normal(0, 1, size=(490, 2))
anom = rng.uniform(-6, 6, size=(10, 2))
X = np.vstack([normal, anom])
y = np.r_[np.zeros(len(normal)), np.ones(len(anom))]    # 1 = anomaly (2% of data)
print(f'{int(y.sum())} anomalies out of {len(X)} points ({100*y.mean():.0f}%)')

## 2 — Three anomaly scores

z-score (distance from the mean in std units), kNN distance (to the k-th neighbor), and a simplified
isolation score (how few random splits isolate a point).

In [ ]:
def zscore_anomaly(X):
    return np.linalg.norm((X - X.mean(0)) / X.std(0), axis=1)

def knn_anomaly(X, k=5):
    D = np.linalg.norm(X[:, None] - X[None, :], axis=2)
    D.sort(axis=1)
    return D[:, k]                                       # distance to the k-th neighbor

def isolation_score(X, n_trees=100, seed=0):
    r = np.random.default_rng(seed)
    n = len(X)
    depths = np.zeros(n)
    for _ in range(n_trees):
        idx = np.arange(n); depth = np.zeros(n); active = np.ones(n, bool)
        lo, hi = X.min(0), X.max(0)
        bounds = {tuple(idx): (lo.copy(), hi.copy())}
        # simplified: split the whole space repeatedly, count splits until each point is alone-ish
        for d in range(1, 12):
            f = r.integers(0, X.shape[1])
            t = r.uniform(X[active, f].min(), X[active, f].max()) if active.sum() > 1 else 0
            left = active & (X[:, f] < t)
            # points in the smaller partition are 'more isolated' -> stop deepening them
            small = left if left.sum() < (active & ~left).sum() else (active & ~left)
            depth[small & (depth == 0)] = d
            active = active & ~small
            if active.sum() <= 1:
                depth[active & (depth == 0)] = d
                break
        depths += depth
    return -depths / n_trees                              # short path (small depth) -> high anomaly

for name, score in [('z-score', zscore_anomaly(X)), ('kNN', knn_anomaly(X)), ('isolation', isolation_score(X))]:
    top10 = set(np.argsort(-score)[:10])
    recall = len(top10 & set(np.where(y == 1)[0])) / 10
    print(f'{name:10s}: recall@10 = {recall:.1f}')

## 3 — Accuracy lies; use precision/recall at a threshold

A detector that calls everything 'normal' is 98% accurate here yet useless. We threshold the z-score
and trace precision vs recall as the cutoff moves.

In [ ]:
print('accuracy of a "call everything normal" detector:', f'{(y==0).mean():.3f}  <- useless!')
score = zscore_anomaly(X)
print('\nthreshold   precision   recall   #flagged')
for thr in np.percentile(score, [90, 95, 98, 99]):
    flag = score >= thr
    prec = (flag & (y==1)).sum() / max(flag.sum(), 1)
    rec = (flag & (y==1)).sum() / (y==1).sum()
    print(f'{thr:7.2f}     {prec:.2f}        {rec:.2f}      {flag.sum()}')
print('\nLower threshold -> higher recall but more false alarms. Pick by your alert budget.')

### Validate: accuracy is useless; recall-at-alert-rate is what matters

A detector that calls *everything* normal scores near-perfect accuracy yet catches zero
anomalies. The meaningful metric is recall at a fixed alert rate (the top-x% by score). We
confirm the accuracy trap and that the z-score catches most of these far-flung anomalies at
a 5% alert budget.

In [ ]:
acc_trivial = (y == 0).mean()
print(f'"call everything normal" accuracy: {acc_trivial:.3f} (catches {0} anomalies)')
assert acc_trivial > 0.95 and (np.zeros_like(y)[y == 1]).sum() == 0, 'high accuracy, zero recall'

def recall_at(score, rate):
    thr = np.percentile(score, 100 * (1 - rate))
    flag = score >= thr
    return (flag & (y == 1)).sum() / (y == 1).sum()

s = zscore_anomaly(X)
for rate in [0.02, 0.05, 0.10]:
    print(f'alert rate {rate:.0%}: recall = {recall_at(s, rate):.2f}')
assert recall_at(s, 0.10) >= recall_at(s, 0.02), 'flagging more can only raise recall'
assert recall_at(s, 0.05) >= 0.7, 'z-score catches most far anomalies at a 5% alert rate'
print('\n✅ accuracy misleads on rare events; evaluate recall at your alert budget')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **accuracy on rare events** | near-perfect and useless; use recall at an alert rate (verified) |
| **threshold choice** | the alert rate is a precision/recall dial set by how many alerts you can action |
| **training-set contamination** | anomalies in the "normal" training data raise the threshold |
| **distribution assumptions** | z-score fails on non-Gaussian data; pick the method to match |
| **concept drift** | "normal" shifts over time; re-fit the detector |

Demo: all three scorers separate obvious anomalies but rest on different assumptions.

In [ ]:
# The three scores agree on obvious anomalies but not on subtle ones — which is why you
# pick the method by the DATA. We check that all three rank the injected anomalies above
# the median normal point (they all catch the easy cases).
for name, sc in [('z-score', zscore_anomaly(X)), ('kNN', knn_anomaly(X)), ('isolation', isolation_score(X))]:
    anom_median = np.median(sc[y == 1])
    normal_median = np.median(sc[y == 0])
    print(f'{name:10s}: median anomaly score {anom_median:.2f} vs normal {normal_median:.2f}')
    assert anom_median > normal_median, f'{name} should score anomalies above normal points'
print('\nAll three separate obvious anomalies; on subtle ones they disagree -> match method to data')
print('(z-score assumes Gaussian; kNN needs density; isolation handles high dimensions).')

## ✏️ Your turn

**Exercise.** Implement `zscore(X)` (per-point Euclidean norm of the standardized features) and
`recall_at_alert_rate(score, y, rate)` — flag the top `rate` fraction of points by score and return
the recall (fraction of true anomalies caught). This is how you evaluate a detector under a fixed
alert budget.

In [ ]:
def zscore(X):
    # TODO(you): standardize features, return the Euclidean norm per point
    return ...

def recall_at_alert_rate(score, y, rate):
    # TODO(you): flag the top `rate` fraction by score; return recall over the true anomalies (y==1)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
s = zscore(X)
assert np.allclose(s, zscore_anomaly(X))
# flagging more points can only help recall (monotonic)
assert recall_at_alert_rate(s, y, 0.10) >= recall_at_alert_rate(s, y, 0.02)
assert 0.0 <= recall_at_alert_rate(s, y, 0.05) <= 1.0
# z-score catches most of these far-flung anomalies even at a 5% alert rate
assert recall_at_alert_rate(s, y, 0.05) >= 0.7
print('\u2713 zscore and recall@alert-rate are correct')

<details>
<summary>Solution</summary>

```python
def zscore(X):
    return np.linalg.norm((X - X.mean(0)) / X.std(0), axis=1)

def recall_at_alert_rate(score, y, rate):
    k = max(1, int(rate * len(score)))
    flagged = set(np.argsort(-score)[:k])
    caught = len(flagged & set(np.where(y == 1)[0]))
    return caught / (y == 1).sum()
```

Reporting recall at a fixed alert rate (or precision@k) is the honest way to score an anomaly
detector — it bakes in the real constraint that the team can only investigate so many alerts.

</details>

## Key takeaways

- **Anomalies are rare, so accuracy is a trap:** a do-nothing detector scores 98%+ and
  catches nothing (verified).
- **Score anomaly-ness and evaluate at an alert rate:** recall at the top-x% is the
  operational metric (verified).
- **Three scorers, three assumptions:** z-score (Gaussian), kNN (density), isolation
  (high-dim) — all catch obvious anomalies, disagree on subtle ones (demo).
- **Match the method to the data** and tune the threshold to your alert budget.